In [ ]:
import pandas as pd
import numpy as np

# ─── SET THIS EACH YEAR ───────────────────────────────────────────────────────
current_year = 2026   # The EDC lineup we are trying to predict
# ─────────────────────────────────────────────────────────────────────────────

## Data Transformation Explained

To predict if an artist plays in a specific year (e.g., 2026), we need to restructure our data.
Currently, we have **one row per artist** with a list of years played.
We need **one row per artist-year combination** (the "Long Format").

**Transformation Steps:**
1.  **Parse "Years Played":** Convert the string `"2022, 2024"` into binary columns: `played_2022`, `played_2023`, etc.
2.  **Clean Agency Data:** Identify which artists are managed by "Insomniac" (the festival organizer).
3.  **Create "Lag Features":** For every target year (e.g., 2025), looking backwards:
    *   Did they play 1 year ago? (2024)
    *   Did they play 2 years ago? (2023)
    *   How many total times have they played *before* this year?
4.  **Target Variable:** Did they actually play in the target year? (0 or 1)

In [ ]:
# Load datasets
try:
    df_stats = pd.read_csv('../data/main/COMPLETE_edc_artist_and_stats.csv')
    df_residency = pd.read_csv(f'../data/extract/{current_year}_vegas_recidency.csv')
    print("Data loaded successfully.")
    display(df_stats.head(3))
except FileNotFoundError:
    print("Files not found. Please check the paths.")

In [ ]:
# Parse "Years Played" into Binary Columns
# ---------------------------------------------------------

HISTORY_YEARS = list(range(2022, current_year))  # e.g. [2022,2023,2024,2025] when current_year=2026

def parse_years(x):
    if pd.isna(x):
        return []
    # Clean the string: remove quotes, split by comma, convert to int
    return [int(y.strip()) for y in str(x).replace('"', '').split(',') if y.strip().isdigit()]

# Apply the parser
df_stats['years_played_list'] = df_stats['years_played'].apply(parse_years)

# Create binary columns for each year
for year in HISTORY_YEARS:
    col_name = f'played_{year}'
    df_stats[col_name] = df_stats['years_played_list'].apply(lambda x: 1 if year in x else 0)

# Check our work
cols_to_show = ['artist', 'years_played'] + [f'played_{y}' for y in HISTORY_YEARS]
display(df_stats[cols_to_show].head())

# 2. Clean Agency Data (Insomniac Flag)
# ---------------------------------------------------------
# Create a simple binary flag for the main organizer
df_stats['is_insomniac'] = df_stats['agency'].fillna('').str.lower().str.contains('insomniac').astype(int)

# Log Scale for Followers (Handles huge range 100 vs 10M)
df_stats['log_followers'] = np.log1p(df_stats['followers'].fillna(0))

print(f"Insomniac Artists found: {df_stats['is_insomniac'].sum()}")
df_stats.head()

In [ ]:
# Create Training Set (Simulate current_year - 1)
# ---------------------------------------------------------
# We pretend we are in late (current_year - 2) trying to predict the (current_year - 1) lineup.
# This allows us to have a "Ground Truth" (who actually played) to train our model.

training_data = []
train_target_year = current_year - 1  # e.g., 2025 when predicting 2026

for idx, row in df_stats.iterrows():
    # INPUTS: What did we know BEFORE train_target_year?
    played_prev_year = row[f'played_{train_target_year - 1}']
    played_2_years_ago = row[f'played_{train_target_year - 2}']
    played_3_years_ago = row[f'played_{train_target_year - 3}']
    
    # Feature: Total past appearances (up to train_target_year - 1)
    total_past_appearances = sum(row[f'played_{y}'] for y in range(2022, train_target_year))
    
    # Feature: Consecutive years played recently (burnout indicator)
    consecutive = 0
    if played_prev_year == 1:
        consecutive = 1
        if played_2_years_ago == 1:
            consecutive = 2
            
    # TARGET: Did they ACTUALLY play in train_target_year?
    played_target_year = row[f'played_{train_target_year}']
    
    # Combine everything into a training row
    training_data.append({
        'artist': row['artist'],
        'played_prev_year': played_prev_year,
        'played_2_years_ago': played_2_years_ago,
        'played_3_years_ago': played_3_years_ago,
        'total_past_appearances': total_past_appearances,
        'consecutive_years': consecutive,
        'is_insomniac': row['is_insomniac'],
        'log_followers': row['log_followers'],
        'streams': row['streams'],
        'followers_growth': row.get('followers_growth', 0.0),
        'streams_growth': row.get('streams_growth', 0.0),
        f'target_played_{train_target_year}': played_target_year
    })

df_train = pd.DataFrame(training_data)

print(f"Training Data Created: {df_train.shape[0]} samples")
display(df_train.head())

In [ ]:
# Train the Model (Random Forest)
# ---------------------------------------------------------
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Define Features and Target
features = ['played_prev_year', 'played_2_years_ago', 'played_3_years_ago', 
            'total_past_appearances', 'consecutive_years', 
            'is_insomniac', 'log_followers', 'streams',
            'followers_growth', 'streams_growth']
target = f'target_played_{current_year - 1}'

# Initialize Model
# n_estimators=100: Build 100 decision trees
# class_weight='balanced': Handle the fact that most artists DON'T play (imbalanced data)
model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, class_weight='balanced')

# Fit Model on the training data
model.fit(df_train[features], df_train[target])

print("Model Trained Successfully!")

# Check Feature Importance (What did the model learn?)
importances = pd.DataFrame({'feature': features, 'importance': model.feature_importances_})
importances = importances.sort_values('importance', ascending=False)
display(importances)

In [ ]:
# Predict current_year (The Future)
# ---------------------------------------------------------
# Now we gather the data as it stands TODAY (for the current_year festival).

data_predict = []

for idx, row in df_stats.iterrows():
    # INPUTS for current_year Prediction (shifted forward by 1 year vs. training)
    played_prev_year = row[f'played_{current_year - 1}']
    played_2_years_ago = row[f'played_{current_year - 2}']
    played_3_years_ago = row[f'played_{current_year - 3}']
    
    # Total past (all history years)
    total_past_appearances = sum(row[f'played_{y}'] for y in HISTORY_YEARS)
    
    consecutive = 0
    if played_prev_year == 1:
        consecutive = 1
        if played_2_years_ago == 1:
            consecutive = 2
    
    data_predict.append({
        'artist': row['artist'],
        'played_prev_year': played_prev_year,
        'played_2_years_ago': played_2_years_ago,
        'played_3_years_ago': played_3_years_ago,
        'total_past_appearances': total_past_appearances,
        'consecutive_years': consecutive,
        'is_insomniac': row['is_insomniac'],
        'log_followers': row['log_followers'],
        'streams': row['streams'],
        'followers_growth': row.get('followers_growth', 0.0),
        'streams_growth': row.get('streams_growth', 0.0),
    })

df_predict = pd.DataFrame(data_predict)

# Make Probability Predictions (0 to 1)
# predict_proba returns [prob_0, prob_1] -> we want column 1
df_predict['raw_probability'] = model.predict_proba(df_predict[features])[:, 1]

print("Predictions generated. Sample:")
display(df_predict[['artist', 'raw_probability']].sort_values('raw_probability', ascending=False).head())

In [ ]:
# Apply Logic Controls & Residency Filter
# ---------------------------------------------------------

# Load Residency List
residency_artists = df_residency['artist'].str.lower().str.strip().unique()

# Define "Big Artist" Threshold (Top 25% most followed)
big_artist_threshold = df_predict['log_followers'].quantile(0.75)

def adjust_score(row):
    score = row['raw_probability']
    artist_name = str(row['artist']).lower().strip()
    
    # 1. BOOST: Insomniac Agency (+45%)
    if row['is_insomniac'] == 1:
        score = score * 1.45
        
    # 2. PATTERN: "2 Years On, 1 Year Off" for Big Artists
    # Big artists usually play 2 years then skip 1.
    if row['log_followers'] > big_artist_threshold:
        if row['consecutive_years'] == 1:
            # Played last year but not the year before -> likely start of Year 2 cycle -> Boost
            score = score * 1.25 
        elif row['consecutive_years'] >= 2:
            # Played 2+ years in a row -> likely taking a break -> Penalty
            score = score * 0.75

    # 3. PENALTY: Vegas Residency (-30%)
    has_residency = artist_name in residency_artists
    if has_residency:
        score = score * 0.7
        
    # Cap at 1.0 (100%)
    return min(score, 1.0), has_residency

# Apply adjustments
df_predict[['final_probability', 'has_residency']] = df_predict.apply(adjust_score, axis=1, result_type='expand')

# Round probabilities for clarity
df_predict['raw_probability'] = df_predict['raw_probability'].round(5)
df_predict['final_probability'] = df_predict['final_probability'].round(5)

# Sort and Show the Lineup Prediction
final_prediction = df_predict.sort_values('final_probability', ascending=False)

# 4. CAP PREDICTION: Limit positive results to around 285 artists
TARGET_LINEUP_SIZE = 285
if len(final_prediction) >= TARGET_LINEUP_SIZE:
    dynamic_threshold = final_prediction['final_probability'].iloc[TARGET_LINEUP_SIZE - 1]
else:
    dynamic_threshold = 0.5 

# Add binary prediction based on dynamic threshold
final_prediction['result'] = (final_prediction['final_probability'] >= dynamic_threshold).astype(int)

print(f"--- TOP 50 PREDICTED ARTISTS FOR EDC {current_year} ---")
cols_output = ['artist', 'final_probability', 'result', 'has_residency', 'is_insomniac', 'played_prev_year', 'consecutive_years', 'total_past_appearances']
display(final_prediction[cols_output].head(50))

# Save to CSV
output_path = f"../data/result_prediction/edc_{current_year}_prediction.csv"
final_prediction.to_csv(output_path, index=False)
print(f"Saved prediction to {output_path}")